# 🎮 Glitch Media — Multi-Agentic AI Pipeline

> Otomatisasi media berita game: Scraping → Screening → Caption → Image → HITL → Upload

Jalankan setiap sel secara berurutan. Pipeline akan berhenti di **Cell 6** untuk menunggu keputusan ACC/REJECT.

---
## Cell 1 — Import & Load Environment

In [ ]:
import sys
import logging
from pathlib import Path

# Pastikan project root ada di sys.path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import config (otomatis load .env)
from config import settings

# Setup logging
settings.LOGS_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(name)s] %(levelname)s: %(message)s',
    handlers=[
        logging.FileHandler(settings.LOG_FILE, encoding='utf-8'),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger('main')

# Import pipeline modules
from pipeline.state import PipelineState
from pipeline import orchestrator
from validation import human_validation

print('✅ Import & environment loaded successfully.')
print(f'📁 Project root: {PROJECT_ROOT}')
print(f'🤖 LLM Model: {settings.LLM_MODEL}')
print(f'🎨 Image Gen: {settings.IMAGE_GEN_PROVIDER} ({settings.IMAGE_GEN_MODEL})')

---
## Cell 2 — Inisialisasi Pipeline State

Buat `PipelineState` dengan `run_id` unik. Atau load dari checkpoint jika ingin resume.

In [ ]:
from datetime import datetime, timezone

# ── Buat run baru ────────────────────────────────────────────────────────
run_id = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
state = PipelineState(run_id=run_id)

# ── ATAU: Resume dari checkpoint ─────────────────────────────────────────
# Uncomment baris di bawah untuk resume dari checkpoint:
# state = human_validation.load_checkpoint('20240101_120000')  # ganti dengan run_id

print(f'🆔 Run ID: {state.run_id}')
print(f'📊 State initialized: raw_data={state.raw_data is not None}, '
      f'filtered={state.filtered_data is not None}, '
      f'caption_x={state.caption_x is not None}, '
      f'caption_ig={state.caption_ig is not None}')

# Sumber berita yang akan di-scrape
sources = settings.DEFAULT_SOURCES
print(f'📰 Sources: {sources}')

---
## Cell 3 — Stage 1-2: Scraping + Screening

Sequential: Scraping → Screening (dedup + quality filter)

In [ ]:
state = orchestrator.run_stage_1_2(state, sources)

print(f'\n📥 Stage 1 (Scraping): {len(state.raw_data.items)} raw items')
print(f'🔍 Stage 2 (Screening): {len(state.filtered_data.items)} filtered items')

# Preview filtered items
print('\n--- Filtered Items ---')
for i, item in enumerate(state.filtered_data.items, 1):
    print(f'{i}. [{item.source}] {item.title} (score: {item.quality_score:.2f})')
    print(f'   {item.summary}')
    print()

---
## Cell 4 — Stage 3: Caption X & IG (Parallel Fork)

Caption X dan Caption IG berjalan paralel — tidak saling bergantung.

In [ ]:
state = orchestrator.run_stage_3_fork(state)

if state.caption_x:
    print(f'🐦 Caption X untuk: {state.caption_x.news_ref_title}')
    print(f'   {state.caption_x.caption_long[:200]}...')
    print()

if state.caption_ig:
    print(f'📸 Caption IG untuk: {state.caption_ig.news_ref_title}')
    print(f'   {state.caption_ig.caption_short[:200]}...')

---
## Cell 5 — Stage 4: Image Generation

Sequential — tergantung pada caption IG (stage 3b).

In [ ]:
state = orchestrator.run_stage_4(state)

if state.image_result:
    print(f'🖼️ Image generated: {state.image_result.image_path}')
    print(f'   Prompt: {state.image_result.image_prompt[:150]}...')
    
    # Tampilkan gambar di notebook
    from IPython.display import display, Image as IPImage
    try:
        display(IPImage(filename=state.image_result.image_path, width=400))
    except Exception as e:
        print(f'   ⚠️ Tidak bisa menampilkan gambar: {e}')

---
## Cell 6 — Stage 5: Human-in-the-Loop Validation

⚠️ **Pipeline BERHENTI di sini** menunggu keputusan ACC/REJECT.

State di-checkpoint sebelum validasi — bisa di-resume jika notebook terputus.

In [ ]:
# Simpan checkpoint sebelum gate HITL
human_validation.save_checkpoint(state)
print(f'💾 Checkpoint saved: data/checkpoints/{state.run_id}.json')
print()

# Tampilkan preview dan minta keputusan
# ⚠️ Pipeline akan BERHENTI di sini sampai tombol ACC/REJECT ditekan
result = human_validation.request_validation(state)

---
## Cell 7 — Stage 6: Upload (jika ACC)

Jika ACC → Upload X dan IG secara paralel.
Jika REJECT → Log rejection, pipeline berhenti.

In [ ]:
if result.status == 'ACC':
    print('✅ Konten di-APPROVE. Memulai upload...')
    print()
    state = orchestrator.run_stage_6_upload(state)
    print()
    print('🎉 Upload selesai!')
else:
    print('❌ Konten di-REJECT.')
    if result.reviewer_note:
        print(f'📝 Catatan reviewer: {result.reviewer_note}')
    print()
    print('💡 Opsi:')
    print('   1. Jalankan ulang dari Cell 4 (re-generate caption)')
    print('   2. Jalankan ulang dari Cell 3 (re-scrape dan re-screen)')
    print('   3. Edit caption secara manual dan jalankan Cell 7 lagi')

---
## Cell 8 — Ringkasan Hasil

In [ ]:
from IPython.display import display, HTML

display(HTML('<h2 style="color: #4CAF50;">📊 Ringkasan Pipeline Run</h2>'))

print(f'🆔 Run ID: {state.run_id}')
print(f'📥 Raw items scraped: {len(state.raw_data.items) if state.raw_data else 0}')
print(f'🔍 Filtered items: {len(state.filtered_data.items) if state.filtered_data else 0}')
print()

if state.caption_x:
    print(f'🐦 Caption X: {state.caption_x.news_ref_title}')
if state.caption_ig:
    print(f'📸 Caption IG: {state.caption_ig.news_ref_title}')
if state.image_result:
    print(f'🖼️ Image: {state.image_result.image_path}')
print()

if state.validation:
    print(f'✅ Validation: {state.validation.status}')
print()

print('--- Run Log ---')
for log_entry in state.logs:
    print(f'  {log_entry}')